## Accelerate Inference: Neural Network Pruning

In [1]:
import os
import numpy as np
import cv2
import pickle
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# untar
!ls
!tar -xvzf dataset.tar.gz
# load train
train_images = pickle.load(open('train_images.pkl', 'rb'))
train_labels = pickle.load(open('train_labels.pkl', 'rb'))
# load val
val_images = pickle.load(open('val_images.pkl', 'rb'))
val_labels = pickle.load(open('val_labels.pkl', 'rb'))

dataset.tar.gz	train_images.pkl  val_images.pkl
sample_data	train_labels.pkl  val_labels.pkl
train_images.pkl
train_labels.pkl
val_images.pkl
val_labels.pkl


In [4]:
train_images = torch.tensor(train_images, dtype=torch.float32)
val_images = torch.tensor(val_images, dtype=torch.float32)

train_images = train_images.permute(0, 3, 1, 2)
val_images = val_images.permute(0, 3, 1, 2)

In [5]:
train_dataset = TensorDataset(train_images,
                              torch.tensor(train_labels.squeeze(), dtype=torch.long))
val_dataset = TensorDataset(val_images,
                            torch.tensor(val_labels.squeeze(), dtype=torch.long))

In [6]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [7]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()

        self.model = nn.Sequential(
            # First block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Second block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Flatten layer
            nn.Flatten(),

            # Fully connected block: Dense -> ReLU -> Dropout -> Dense -> Softmax
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 5),
        )

    def forward(self, x):
        return self.model(x)

In [8]:
model = ConvNet()

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)

In [9]:
model = model.to(device)
summary(model, input_size=(3, 25, 25))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 25, 25]             896
              ReLU-2           [-1, 32, 25, 25]               0
            Conv2d-3           [-1, 32, 23, 23]           9,248
              ReLU-4           [-1, 32, 23, 23]               0
         MaxPool2d-5           [-1, 32, 11, 11]               0
           Dropout-6           [-1, 32, 11, 11]               0
            Conv2d-7           [-1, 64, 11, 11]          18,496
              ReLU-8           [-1, 64, 11, 11]               0
            Conv2d-9             [-1, 64, 9, 9]          36,928
             ReLU-10             [-1, 64, 9, 9]               0
        MaxPool2d-11             [-1, 64, 4, 4]               0
          Dropout-12             [-1, 64, 4, 4]               0
          Flatten-13                 [-1, 1024]               0
           Linear-14                  [

In [10]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()  # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the training loop
    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        optimizer.zero_grad()  # Zero the parameter gradients
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Update tqdm description with current loss and accuracy
        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_accuracy

In [11]:
def validate(model, val_loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the validation loop
    val_loader_tqdm = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():  # Disable gradient calculations for validation
        for inputs, labels in val_loader_tqdm:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Track loss and accuracy
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # Update tqdm description with current validation loss and accuracy
            val_loader_tqdm.set_postfix(loss=val_loss / total, accuracy=100 * correct / total)

    val_accuracy = 100 * correct / total
    val_loss = val_loss / len(val_loader)
    return val_loss, val_accuracy

In [12]:
# Main training loop
num_epochs = 50
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    # Training
    train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, criterion, device)

    # Validation
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)

    # Print epoch results
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')

Epoch 1/50


Epoch [1/50], Train Loss: 1.5180, Train Acc: 30.65%, Val Loss: 1.3774, Val Acc: 40.04%
Epoch 2/50


Epoch [2/50], Train Loss: 1.3634, Train Acc: 40.70%, Val Loss: 1.3022, Val Acc: 43.37%
Epoch 3/50


Epoch [3/50], Train Loss: 1.3135, Train Acc: 43.51%, Val Loss: 1.2640, Val Acc: 46.02%
Epoch 4/50


Epoch [4/50], Train Loss: 1.2748, Train Acc: 46.04%, Val Loss: 1.2343, Val Acc: 46.69%
Epoch 5/50


...

Epoch [46/50], Train Loss: 0.7246, Train Acc: 72.39%, Val Loss: 0.8154, Val Acc: 68.08%
Epoch 47/50


Epoch [47/50], Train Loss: 0.7255, Train Acc: 72.44%, Val Loss: 0.8207, Val Acc: 67.84%
Epoch 48/50


Epoch [48/50], Train Loss: 0.7134, Train Acc: 72.67%, Val Loss: 0.8038, Val Acc: 68.83%
Epoch 49/50


Epoch [49/50], Train Loss: 0.7051, Train Acc: 72.98%, Val Loss: 0.8011, Val Acc: 68.71%
Epoch 50/50


Epoch [50/50], Train Loss: 0.6973, Train Acc: 73.50%, Val Loss: 0.7986, Val Acc: 68.87%


In [13]:
torch.save(model.state_dict(), 'my_model_weights_1.pt', _use_new_zipfile_serialization=False)

### GMP

Gradual Magnitude Pruning (GMP) progressively increases sparsity during training using a polynomial schedule.

**Reference:** Zhu & Gupta (2017) - "To prune, or not to prune: exploring the efficacy of pruning for model compression"

**Key differences from one-shot pruning:**
- Gradually increases sparsity from 0% to target over multiple epochs
- Uses polynomial schedule: $s_t = s_f + (s_i - s_f)(1 - \frac{t-t_0}{n\Delta t})^3$
- Model adapts to increasing sparsity during training

In [14]:
def compute_sparsity(model):
    """Calculate overall sparsity of the model"""
    total_zeros = sum(torch.sum(p == 0).item() for p in model.parameters())
    total_params = sum(p.numel() for p in model.parameters())
    return total_zeros / total_params

def polynomial_schedule(initial_sparsity, final_sparsity, t, t_start, t_end, exponent=3):
    """Compute sparsity at time t using polynomial schedule"""
    if t <= t_start:
        return initial_sparsity
    elif t >= t_end:
        return final_sparsity
    else:
        progress = (t - t_start) / (t_end - t_start)
        return final_sparsity + (initial_sparsity - final_sparsity) * (1 - progress) ** exponent

def apply_magnitude_pruning(model, target_sparsity):
    """Apply global magnitude-based pruning and return mask"""
    mask = {}

    with torch.no_grad():
        # Collect all weights (not biases)
        all_weights = []
        for name, param in model.named_parameters():
            if "weight" in name:
                all_weights.append(param.abs().flatten())

        # Concatenate and calculate global threshold
        all_weights = torch.cat(all_weights)
        threshold = torch.quantile(all_weights, target_sparsity)

        # Apply pruning to each layer and create mask
        for name, param in model.named_parameters():
            if "weight" in name:
                m = (param.abs() > threshold).float().to(device)
                param.mul_(m)
                mask[name] = m

    return mask, threshold

def train_one_epoch_with_pruning(model, train_loader, optimizer, criterion, device, mask=None):
    """Training function with mask enforcement"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        optimizer.zero_grad()
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        # CRITICAL: Re-apply mask after optimizer step
        if mask is not None:
            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in mask:
                        param.mul_(mask[name])

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_accuracy

# Load the trained model
model.load_state_dict(torch.load('my_model_weights_1.pt'))
model = model.to(device)

# GMP hyperparameters
num_epochs = 65
initial_sparsity = 0.0
final_sparsity = 0.96
prune_start_epoch = 5
prune_end_epoch = 45
prune_frequency = 2

# Reset optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)

print("="*70)
print("Gradual Magnitude Pruning Configuration:")
print(f"  Total epochs: {num_epochs}")
print(f"  Pruning window: epochs {prune_start_epoch} to {prune_end_epoch}")
print(f"  Target sparsity: {initial_sparsity:.2%} -> {final_sparsity:.2%}")
print(f"  Pruning frequency: every {prune_frequency} epochs")
print("="*70)

best_val_accuracy = 0.0
mask = None  # Initialize mask

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # Calculate current target sparsity
    current_sparsity_target = polynomial_schedule(
        initial_sparsity, final_sparsity,
        epoch, prune_start_epoch, prune_end_epoch
    )

    # Apply pruning at specified frequency
    if epoch >= prune_start_epoch and epoch % prune_frequency == 0:
        print(f"  Applying pruning to target sparsity: {current_sparsity_target:.4f}")
        mask, threshold = apply_magnitude_pruning(model, current_sparsity_target)
        actual_sparsity = compute_sparsity(model)
        print(f"  Threshold: {threshold:.6f}")
        print(f"  Actual sparsity after pruning: {actual_sparsity:.4f}")

    # Training with mask enforcement
    train_loss, train_accuracy = train_one_epoch_with_pruning(
        model, train_loader, optimizer, criterion, device, mask
    )

    # Validation
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)

    # Calculate current sparsity and score
    current_sparsity = compute_sparsity(model)
    score = (val_accuracy/100 + current_sparsity) / 2 if val_accuracy > 60 else 0

    # Print epoch results
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
    print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
    print(f'  Sparsity: {current_sparsity:.4f}, Score: {score:.4f}')

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        print(f" New best validation accuracy: {val_accuracy:.2f}%")

final_sparsity = compute_sparsity(model)
final_val_loss, final_val_accuracy = validate(model, val_loader, criterion, device)
final_score = (final_val_accuracy/100 + final_sparsity) / 2 if final_val_accuracy > 60 else 0

print(f"\n{'='*70}")
print(f"Final Results:")
print(f"  Validation Accuracy: {final_val_accuracy:.2f}%")
print(f"  Sparsity: {final_sparsity:.4f}")
print(f"  Score: {final_score:.4f}")
print(f"{'='*70}\n")

Gradual Magnitude Pruning Configuration:
  Total epochs: 65
  Pruning window: epochs 5 to 45
  Target sparsity: 0.00% -> 96.00%
  Pruning frequency: every 2 epochs

Epoch 1/65


  Train Loss: 0.6900, Train Acc: 73.54%
  Val Loss: 0.8101, Val Acc: 69.11%
  Sparsity: 0.0000, Score: 0.3455
 New best validation accuracy: 69.11%

Epoch 2/65


  Train Loss: 0.6831, Train Acc: 73.78%
  Val Loss: 0.8095, Val Acc: 68.08%
  Sparsity: 0.0000, Score: 0.3404

Epoch 3/65


  Train Loss: 0.6715, Train Acc: 74.51%
  Val Loss: 0.7918, Val Acc: 69.43%
  Sparsity: 0.0000, Score: 0.3471
 New best validation accuracy: 69.43%

Epoch 4/65


  Train Loss: 0.6695, Train Acc: 74.37%
  Val Loss: 0.8102, Val Acc: 68.67%
  Sparsity: 0.0000, Score: 0.3434

Epoch 5/65


...

  Train Loss: 0.7622, Train Acc: 70.74%
  Val Loss: 0.7703, Val Acc: 70.30%
  Sparsity: 0.9589, Score: 0.8309

Epoch 63/65
  Applying pruning to target sparsity: 0.9600
  Threshold: 0.002392
  Actual sparsity after pruning: 0.9589


  Train Loss: 0.7527, Train Acc: 71.01%
  Val Loss: 0.7748, Val Acc: 69.47%
  Sparsity: 0.9589, Score: 0.8268

Epoch 64/65


  Train Loss: 0.7569, Train Acc: 70.98%
  Val Loss: 0.7660, Val Acc: 70.81%
  Sparsity: 0.9589, Score: 0.8335

Epoch 65/65
  Applying pruning to target sparsity: 0.9600
  Threshold: 0.002135
  Actual sparsity after pruning: 0.9589


  Train Loss: 0.7582, Train Acc: 70.44%
  Val Loss: 0.7615, Val Acc: 69.98%
  Sparsity: 0.9589, Score: 0.8293



Final Results:
  Validation Accuracy: 69.98%
  Sparsity: 0.9589
  Score: 0.8293



In [15]:
# val_loss, val_accuracy = validate(model, val_loader, criterion, device)

In [16]:
# val_accuracy

In [17]:
# torch.save(model.state_dict(), 'my_model_weights_2.pt', _use_new_zipfile_serialization=False)
torch.save(model.state_dict(), 'my_model_weights_2.pt', _use_new_zipfile_serialization=False)
print("Model saved as my_model_weights_2.pt")
print(f"Final Sparsity: {compute_sparsity(model):.4f}")
print(f"Final Accuracy: {final_val_accuracy:.2f}%")
print(f"Final Score: {final_score:.4f}")

Model saved as my_model_weights_2.pt
Final Sparsity: 0.9589
Final Accuracy: 69.98%
Final Score: 0.8293


In [18]:
# Different configurations to try hyperparameter tuning
configurations = [
    {'num_epochs': 60, 'final_sparsity': 0.955, 'prune_start': 5, 'prune_end': 40},
    {'num_epochs': 65, 'final_sparsity': 0.96, 'prune_start': 5, 'prune_end': 45},
    {'num_epochs': 70, 'final_sparsity': 0.965, 'prune_start': 5, 'prune_end': 50},
    {'num_epochs': 75, 'final_sparsity': 0.97, 'prune_start': 5, 'prune_end': 55},
    {'num_epochs': 50, 'final_sparsity': 0.94, 'prune_start': 5, 'prune_end': 35},
]

best_score = 0
best_config = None
best_state = None

for config in configurations:
    print(f"\n{'='*70}")
    print(f"Testing config: {config}")
    print(f"{'='*70}")

    # Reload original trained model
    model.load_state_dict(torch.load('my_model_weights_1.pt'))
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)
    mask = None

    for epoch in range(config['num_epochs']):
        current_sparsity_target = polynomial_schedule(
            0.0, config['final_sparsity'],
            epoch, config['prune_start'], config['prune_end']
        )

        # Apply pruning and update mask
        if epoch >= config['prune_start'] and epoch % 2 == 0:
            mask, threshold = apply_magnitude_pruning(model, current_sparsity_target)

        # Training with mask enforcement
        train_loss, train_acc = train_one_epoch_with_pruning(
            model, train_loader, optimizer, criterion, device, mask  # Pass mask
        )

        # Periodic validation
        if (epoch + 1) % 10 == 0:
            val_loss, val_acc = validate(model, val_loader, criterion, device)
            sparsity = compute_sparsity(model)
            print(f"  Epoch {epoch+1}: Val Acc: {val_acc:.2f}%, Sparsity: {sparsity:.4f}")

    # Final evaluation
    final_sparsity = compute_sparsity(model)
    _, final_acc = validate(model, val_loader, criterion, device)
    score = (final_acc/100 + final_sparsity) / 2 if final_acc > 60 else 0

    print(f"\nResults:")
    print(f"  Val Acc: {final_acc:.2f}%")
    print(f"  Sparsity: {final_sparsity:.4f}")
    print(f"  Score: {score:.4f}")

    if score > best_score:
        best_score = score
        best_config = config
        best_state = model.state_dict().copy()  # Save best model state
        print(f" New best score!")

print(f"\n{'='*70}")
print(f"Best Configuration:")
print(f"  Config: {best_config}")
print(f"  Score: {best_score:.4f}")
print(f"{'='*70}")

if best_state is not None:
    torch.save(best_state, 'my_model_weights_2_best.pt', _use_new_zipfile_serialization=False)
    print("\nBest model saved as my_model_weights_2_best.pt")


Testing config: {'num_epochs': 60, 'final_sparsity': 0.955, 'prune_start': 5, 'prune_end': 40}


  Epoch 10: Val Acc: 69.58%, Sparsity: 0.2249


  Epoch 20: Val Acc: 69.98%, Sparsity: 0.7170


  Epoch 30: Val Acc: 68.44%, Sparsity: 0.9154


  Epoch 40: Val Acc: 68.59%, Sparsity: 0.9537


...

  Epoch 20: Val Acc: 70.53%, Sparsity: 0.7680


  Epoch 30: Val Acc: 69.27%, Sparsity: 0.9269


  Epoch 40: Val Acc: 70.06%, Sparsity: 0.9389


  Epoch 50: Val Acc: 70.10%, Sparsity: 0.9389



Results:
  Val Acc: 70.10%
  Sparsity: 0.9389
  Score: 0.8199

Best Configuration:
  Config: {'num_epochs': 75, 'final_sparsity': 0.97, 'prune_start': 5, 'prune_end': 55}
  Score: 0.8308

Best model saved as my_model_weights_2_best.pt
